In [1]:
import os

In [2]:
os.chdir('../')

In [3]:
from src.end_to_end_deep_learning_project_yt.entity.config_entity import (
    Config, Params
    , DataIngestionConfig, PrepareBaseModelConfig
)

In [4]:
from pydantic import BaseModel, ConfigDict
from pathlib import Path

class PrepareCallbacksConfig(BaseModel):
    model_config = ConfigDict(extra='forbid')

    root_dir: Path
    tensorboard_root_log_dir: Path
    checkpoint_model_filepath: Path


class Config(BaseModel):
    model_config = ConfigDict(extra='forbid')

    artifacts_root: Path
    data_ingestion: DataIngestionConfig
    prepare_base_model: PrepareBaseModelConfig
    prepare_callbacks: PrepareCallbacksConfig

In [5]:
from src.end_to_end_deep_learning_project_yt.constants import constants
from src.end_to_end_deep_learning_project_yt.utils.common import read_yaml, create_directories

In [6]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = constants.CONFIG_FILE_PATH,
        params_filepath = constants.PARAMS_FILE_PATH
    ):
        self.config = read_yaml(config_filepath, Config)
        self.params = read_yaml(params_filepath, Params)

        create_directories([self.config.artifacts_root])

    def get_prepare_callback_config(self) -> PrepareCallbacksConfig:
        prepare_callback_config = self.config.prepare_callbacks

        model_ckpt_dir = os.path.dirname(prepare_callback_config.checkpoint_model_filepath)

        create_directories([
            Path(model_ckpt_dir),
            Path(prepare_callback_config.tensorboard_root_log_dir)
        ])

        return prepare_callback_config

In [7]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf
import time

I0000 00:00:1794070296.536984   78817 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [8]:
class PrepareCallback:
    def __init__(self, config: PrepareCallbacksConfig):
        self.config = config

    @property
    def _create_tb_callbacks(self):
        timestamp = time.strftime('%Y-%m-%d-%H-%M-%S')
        tb_running_log_dir = os.path.join(
            self.config.tensorboard_root_log_dir,
            f'tb_log_at_{timestamp}'
        )

        return tf.keras.callbacks.TensorBoard(log_dir=tb_running_log_dir)

    @property
    def _create_ckpt_callbacks(self):
        return tf.keras.callbacks.ModelCheckpoint(
            filepath=self.config.checkpoint_model_filepath,
            save_best_only=True
        )

    def get_tb_ckpt_callbacks(self):
        return [
            self._create_tb_callbacks
            , self._create_ckpt_callbacks
        ]

In [9]:
try:
    config = ConfigurationManager()
    prepare_callbacks_config = config.get_prepare_callback_config()
    prepare_callbacks = PrepareCallback(config=prepare_callbacks_config)
    callback_list = prepare_callbacks.get_tb_ckpt_callbacks()
except Exception as e:
    raise e

[2026-11-07 16:51:38,103]: INFO - common - yaml file: config/config.yaml loaded successfully
[2026-11-07 16:51:38,104]: INFO - common - yaml file: params.yaml loaded successfully
[2026-11-07 16:51:38,105]: INFO - common - created directory at: artifacts
[2026-11-07 16:51:38,105]: INFO - common - created directory at: artifacts/prepare_callbacks/checkpoint_dir
[2026-11-07 16:51:38,105]: INFO - common - created directory at: artifacts/prepare_callbacks/tensorboard_log_dir


In [10]:
import time

timestamp = time.strftime('%Y-%m-%d-%H-%M-%S')
f'tb_log_at_{timestamp}'

'tb_log_at_2026-11-07-16-51-38'